## Homework 6: Decision Trees and Ensemble Learning

Goal: predict `fuel_efficiency_mpg` (regression).

### Data preparation

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

import xgboost as xgb

In [2]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

df = pd.read_csv(data)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

Fill missing values with zeros (only `horsepower` and `acceleration` have any).

In [4]:
df = df.fillna(0)
df.isnull().sum().sum()

np.int64(0)

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

X_train.shape, X_val.shape

((6000, 16), (2000, 16))

### Question 1

Decision tree with `max_depth=1`: which feature is used for the single split?

In [7]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



### Question 2

Random forest with `n_estimators=10`: RMSE on validation.

In [8]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
round(rmse, 3)

1.839

### Question 3

Try `n_estimators` in `[10, 50, 100, 150]`. Which gives the lowest RMSE (3 decimals)?

In [9]:
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)

    scores.append((n, rmse))

df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
df_scores['rmse_3dp'] = df_scores.rmse.round(3)
df_scores

,n_estimators,rmse,rmse_3dp
0,10,1.839383,1.839
1,50,1.772863,1.773
2,100,1.767643,1.768
3,150,1.768109,1.768


### Question 4

For each `max_depth` in `[10, 15, 20, 25]`, try each `n_estimators` and take the mean RMSE.

In [10]:
scores = []

for d in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        scores.append((d, n, rmse))

df_scores = pd.DataFrame(scores, columns=['max_depth', 'n_estimators', 'rmse'])
df_scores.groupby('max_depth').rmse.mean().round(4)

max_depth
10    1.7541
15    1.7837
20    1.7878
25    1.7867
Name: rmse, dtype: float64

### Question 5

Feature importance from a random forest (`n_estimators=10, max_depth=20`).

In [11]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

df_importances = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})
df_importances.sort_values('importance', ascending=False)

,feature,importance
9,model_year,0.295915
15,vehicle_weight,0.203067
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078468
4,engine_displacement,0.061152
0,acceleration,0.055342
14,origin=USA,0.042268
2,drivetrain=Front-wheel drive,0.020108
11,num_doors,0.019364
5,fuel_type=Diesel,0.018045


### Question 6

XGBoost with `eta=0.3` vs `eta=0.1`, 100 rounds each. Which gives the better validation RMSE?

In [12]:
features = list(dv.get_feature_names_out())

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval, 'val')]

In [13]:
def train_xgb(eta):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }

    evals_result = {}
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, evals_result=evals_result,
                      verbose_eval=False)

    return model, evals_result['val']['rmse']

In [14]:
results = {}

for eta in [0.3, 0.1]:
    model, val_rmse = train_xgb(eta)
    results[eta] = val_rmse

    print('eta=%s  final RMSE=%.4f  best RMSE=%.4f (round %d)' % (
        eta, val_rmse[-1], min(val_rmse), int(np.argmin(val_rmse)) + 1))

eta=0.3  final RMSE=1.8265  best RMSE=1.7320 (round 11)
eta=0.1  final RMSE=1.7248  best RMSE=1.7122 (round 40)


### Answers

Read each answer from the output above:

| Q | What to look at |
|---|---|
| 1 | The feature in the `export_text` split (map `origin=...` style names back to the column) |
| 2 | The rounded RMSE |
| 3 | The `n_estimators` row with the lowest RMSE |
| 4 | The `max_depth` with the lowest mean RMSE |
| 5 | The top row among `vehicle_weight`, `horsepower`, `acceleration`, `engine_displacement` |
| 6 | The `eta` with the lower RMSE |